In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics.pairwise import paired_euclidean_distances

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
tail_subset_size = 300
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "tail_subset_size": tail_subset_size,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df_full = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
subset_size = min(tail_subset_size, len(df_full))
df = df_full.tail(subset_size).reset_index(drop=True)

print({
    "full_num_examples": len(df_full),
    "subset_num_examples": len(df),
    "subset_strategy": "deterministic_tail",
    "columns": df.columns.tolist(),
})
print(df.head())


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

embedding_dim = int(emb1.shape[1])
euclidean_distance = paired_euclidean_distances(emb1, emb2)
euclidean_similarity = -euclidean_distance

dist_min = float(euclidean_distance.min())
dist_max = float(euclidean_distance.max())
if dist_max > dist_min:
    predicted_score_0_5 = 5.0 * (dist_max - euclidean_distance) / (dist_max - dist_min)
else:
    predicted_score_0_5 = np.full_like(euclidean_distance, 2.5, dtype=np.float32)

print({
    "embedding_dim": embedding_dim,
    "emb1_shape": emb1.shape,
    "emb2_shape": emb2.shape,
    "euclidean_distance_min": dist_min,
    "euclidean_distance_max": dist_max,
})


In [ ]:
pearson_euclidean_similarity = pearsonr(euclidean_similarity, labels).statistic
spearman_euclidean_similarity = spearmanr(euclidean_similarity, labels).statistic
pearson_rescaled = pearsonr(predicted_score_0_5, labels).statistic
spearman_rescaled = spearmanr(predicted_score_0_5, labels).statistic

nearest_idx = int(np.argmin(euclidean_distance))
farthest_idx = int(np.argmax(euclidean_distance))

nearest_pair_stats = {
    "index": nearest_idx,
    "sentence1": df.loc[nearest_idx, "sentence1"],
    "sentence2": df.loc[nearest_idx, "sentence2"],
    "label": float(df.loc[nearest_idx, "label"]),
    "euclidean_distance": float(euclidean_distance[nearest_idx]),
    "euclidean_similarity": float(euclidean_similarity[nearest_idx]),
    "predicted_score_0_5": float(predicted_score_0_5[nearest_idx]),
}

farthest_pair_stats = {
    "index": farthest_idx,
    "sentence1": df.loc[farthest_idx, "sentence1"],
    "sentence2": df.loc[farthest_idx, "sentence2"],
    "label": float(df.loc[farthest_idx, "label"]),
    "euclidean_distance": float(euclidean_distance[farthest_idx]),
    "euclidean_similarity": float(euclidean_similarity[farthest_idx]),
    "predicted_score_0_5": float(predicted_score_0_5[farthest_idx]),
}

results_df = df.copy()
results_df["euclidean_distance"] = euclidean_distance
results_df["euclidean_similarity"] = euclidean_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5

print({
    "pearson_euclidean_similarity": float(pearson_euclidean_similarity),
    "spearman_euclidean_similarity": float(spearman_euclidean_similarity),
    "pearson_rescaled_0_5": float(pearson_rescaled),
    "spearman_rescaled_0_5": float(spearman_rescaled),
})
print({"nearest_pair_stats": nearest_pair_stats})
print({"farthest_pair_stats": farthest_pair_stats})
print(results_df[["sentence1", "sentence2", "label", "euclidean_distance", "euclidean_similarity", "predicted_score_0_5"]].head(10))


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"subset_strategy: deterministic_tail")
print(f"num_examples_full: {len(df_full)}")
print(f"num_examples_subset: {len(df)}")
print(f"embedding_dimensionality: {embedding_dim}")
print(f"pearson_euclidean_similarity: {pearson_euclidean_similarity:.6f}")
print(f"spearman_euclidean_similarity: {spearman_euclidean_similarity:.6f}")
print(f"pearson_rescaled_0_5: {pearson_rescaled:.6f}")
print(f"spearman_rescaled_0_5: {spearman_rescaled:.6f}")
print(f"nearest_pair_distance: {euclidean_distance[nearest_idx]:.6f}")
print(f"farthest_pair_distance: {euclidean_distance[farthest_idx]:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
